# Getting Historical Chain Constituents with LSEG Tick History REST API and LSEG Data Library for Python

The [LSEG Data Library for Python](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python) `Chain` object returns a chain's constituents, which you can then use with `get_data` and `get_history`. But some bonds chains change its constituent over time because exchanges periodically update which bonds meet the  maturity and coupon criteria for delivery. For instance, newly issued Treasuries are added while older bonds that have fallen below the minimum remaining maturity are removed. 

Examples:
- **0#TYc1=DLV**: CBoT 10-Year US Treasury Note
- **0#FGBMc1=DLV**: Eurex Euro-Bobl (5-year German Govt Bond)
- **0#JGBc1=DLV**: Osaka Exchange 10-Year Japanese Government Bond

See [README.md](./README.md) for more about these chain rules.

The `Chain` object returns **current constituents** because it gets data from the Real-Time platform. To resolve constituents for a specific time, use the [LSEG Tick History REST API](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api), which supports current and past constituents, including inactive, historical-only instruments. 

Once you get the constituents of that specific time from the Tick History, you can then retrieve their historical data with Tick History, the Data Library, or another LSEG API such as [Data Platform APIs (RDP)](https://developers.lseg.com/en/api-catalog/refinitiv-data-platform/refinitiv-data-platform-apis).

#### Note

You can also use the Tick History REST API for the entire workflow. See the [Tick History REST API tutorials](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorials-introduction) for details. This example is for developers who use the Data Library as their primary API.


## 1. Imports

In [2]:
import os
import time
import requests
import lseg.data as ld 


## 2. Configuration

In [3]:
BASE_URL = "https://selectapi.datascope.lseg.com/RestApi/v1"
TIMEOUT = 30  # seconds

#CHAIN_RIC = "0#.BCOM"
CHAIN_RIC = "0#TYc1=DLV"
START_2024 = "2024-05-01T00:00:00.000Z"
END_2024 = "2024-06-30T00:00:00.000Z"

DSS_USERNAME = os.environ.get("DSS_USERNAME", "If not set DSS_USERNAME environment variable, provide your DSS Username")
DSS_PASSWORD = os.environ.get("DSS_PASSWORD", "IF not set DSS_PASSWORD environment variable, provide your DSS Password here")

## 3. Authenticate

In [ ]:
# Reuse this session and its headers for Tick History requests.
session = requests.Session()
session.headers["Content-Type"] = "application/json"
session.headers["Prefer"] = "respond-async"

# Get an access token using the DSS credentials.
try:
    auth_response = session.post(
        f"{BASE_URL}/Authentication/RequestToken",
        json={"Credentials": {"Username": DSS_USERNAME, "Password": DSS_PASSWORD}},
        timeout=TIMEOUT,
    )
    auth_response.raise_for_status()
    print(f"Authentication Tick History response: {auth_response.status_code}")
    token = auth_response.json()["value"]
    if token:
        print("Token obtained")
except requests.exceptions.RequestException as exc:
    raise RuntimeError("DSS authentication request failed") from exc

# Include the token in later API requests.
if token:
    session.headers["Authorization"] = f"Token {token}"

Authentication Tick History response: 200
Token obtained


## 4. Resolve the chain

The API `/Search/HistoricalChainResolution` resolve current and past chain constituents given a Chain RIC. Instruments may be currently active, or inactive 'historical only' instruments.

In [ ]:

def resolve_chain_request(_session, base_url, chain_ric, start, end, timeout):
    """Return the constituents of `chain_ric` between `start` and `end`."""
    print(f"Resolving {chain_ric} from {start} to {end}")

    payload = {
        "Request": {
            "ChainRics": [chain_ric],
            "Range": {"Start": start, "End": end},
        }
    }

    try:
        historical_chain_response = _session.post(
            f"{base_url}/Search/HistoricalChainResolution",
            json=payload,
            timeout=timeout,
        )

        # HTTP 202 = still processing; poll the monitor URL in the Location header until it returns 200
        while historical_chain_response.status_code == 202:
            print("Request is still processing, waiting 5 seconds...")
            time.sleep(5)
            historical_chain_response = _session.get(historical_chain_response.headers["Location"], timeout=timeout)

        # Stop here if the API reports an error.
        historical_chain_response.raise_for_status()
        print(f"Historical Chain Resolution response: {historical_chain_response.status_code}")
        # Only a completed 200 response contains the constituent data.
        if historical_chain_response.status_code == 200:
            constituents = historical_chain_response.json()["value"][0]["Constituents"]
            print(f"Tick History:  {len(constituents)} constituents returned")
            return constituents
    except requests.exceptions.RequestException as exc:
        raise RuntimeError("Historical chain resolution request failed") from exc


In [ ]:
constituents_2024 = resolve_chain_request(session, BASE_URL, CHAIN_RIC, START_2024, END_2024, TIMEOUT)

Resolving 0#TYc1=DLV from 2024-05-01T00:00:00.000Z to 2024-06-30T00:00:00.000Z
Historical Chain Resolution response: 200
Tick History:  18 constituents returned


## 5. Keep valid RICs

In [11]:
rics = [
    item["Identifier"]
    for item in constituents_2024
    if item["Status"] == "Valid" and item["IdentifierType"] == "Ric"
]
rics

['91282CBL4=',
 '91282CCB5=',
 '91282CCS8=',
 '91282CDJ7=',
 '91282CDY4=',
 '91282CEP2=',
 '91282CFF3=',
 '91282CFV8=',
 '91282CGM7=',
 '91282CHC8=',
 '91282CJQ5=',
 '91282CJX0=',
 '91282CKC4=',
 '91282CKF7=',
 '91282CKN0=',
 '91282CKQ3=',
 '91282CKU4=',
 '91282CKW0=']

Please let me remind you again. An application can use the other LSEG Tick History - REST API endpoints to get those constituents historical data without using other LSEG APIs. For example:
- [REST API Tutorial 6: On Demand intraday bars extraction](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorial-6-on-demand-intraday-bars-extraction)
- [REST API Tutorial 4: On Demand tick data extraction](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorial-4-on-demand-tick-data-extraction)

In [9]:
ld.open_session()

<lseg.data.session.Definition object at 0x1bc6b54af60 {name='workspace'}>

In [13]:
#bcom_2025_data = ld.get_history(universe=rics, interval="weekly", start=START, end= END)
bcom_2024_data = ld.get_history(universe=rics, fields=["BID"], start=START_2024, end= END_2024)

bcom_2024_data.head(10)

c:\Project\Code\TRTH_Chain_History\.venv\Lib\site-packages\lseg\data\_tools\_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


BID,91282CBL4=,91282CCB5=,91282CCS8=,91282CDJ7=,91282CDY4=,91282CEP2=,91282CFF3=,91282CFV8=,91282CGM7=,91282CHC8=,91282CJQ5=,91282CJX0=,91282CKC4=,91282CKF7=,91282CKN0=,91282CKQ3=,91282CKU4=,91282CKW0=
Date,,,,,,,,,,,,,,,,,,
2024-05-01,79.9375,82.085938,79.25,79.367188,82.007812,88.148438,87.023438,96.335938,91.773438,90.679688,94.84375,96.21875,97.65625,96.90625,99.84375,<NA>,<NA>,<NA>
2024-05-02,80.28125,82.4375,79.59375,79.726562,82.375,88.53125,87.375,96.710938,92.125,91.054688,95.226562,96.617188,98.0625,97.3125,100.25,<NA>,<NA>,<NA>
2024-05-03,80.78125,82.984375,80.15625,80.273438,82.9375,89.125,87.984375,97.375,92.773438,91.6875,95.765625,97.164062,98.625,97.867188,100.8125,<NA>,<NA>,<NA>
2024-05-06,80.820312,83.03125,80.1875,80.3125,83.0,89.1875,88.054688,97.4375,92.84375,91.773438,95.78125,97.1875,98.648438,97.882812,100.835938,<NA>,<NA>,<NA>
2024-05-07,80.945312,83.15625,80.3125,80.46875,83.148438,89.34375,88.210938,97.601562,93.007812,91.945312,95.90625,97.320312,98.773438,98.023438,100.96875,<NA>,<NA>,<NA>
2024-05-08,80.757812,82.96875,80.125,80.28125,82.929688,89.125,87.992188,97.34375,92.773438,91.695312,95.726562,97.125,98.5625,97.8125,100.757812,99.078125,<NA>,<NA>
2024-05-09,80.945312,83.15625,80.320312,80.46875,83.15625,89.34375,88.210938,97.59375,93.0,91.9375,95.914062,97.3125,98.773438,98.015625,100.96875,99.328125,<NA>,<NA>
2024-05-10,80.742188,82.929688,80.09375,80.25,82.90625,89.0625,87.9375,97.289062,92.71875,91.648438,95.671875,97.070312,98.515625,97.75,100.695312,99.0,<NA>,<NA>
2024-05-13,80.773438,82.976562,80.15625,80.3125,82.976562,89.148438,88.007812,97.367188,92.789062,91.71875,95.71875,97.101562,98.554688,97.789062,100.734375,99.078125,<NA>,<NA>


In [16]:
START_2026 = "2026-05-01T00:00:00.000Z"
END_2026   = "2026-06-30T00:00:00.000Z"

In [22]:
constituents_2026 = resolve_chain_request(session, BASE_URL, CHAIN_RIC, START_2026, END_2026, TIMEOUT)

Resolving 0#TYc1=DLV from 2026-05-01T00:00:00.000Z to 2026-06-30T00:00:00.000Z
Historical Chain Resolution response: 200
Tick History:  19 constituents returned


In [23]:
rics = [
    item["Identifier"]
    for item in constituents_2026
    if item["Status"] == "Valid" and item["IdentifierType"] == "Ric"
]
rics[0:20]

['91282CDJ7=',
 '91282CDY4=',
 '91282CFF3=',
 '91282CFV8=',
 '91282CGM7=',
 '91282CHC8=',
 '91282CHT1=',
 '91282CJJ1=',
 '91282CJZ5=',
 '91282CKQ3=',
 '91282CLF6=',
 '91282CPQ8=',
 '91282CPY1=',
 '91282CQC8=',
 '91282CQF1=',
 '91282CQN4=',
 '91282CQQ7=',
 '91282CQT1=',
 '91282CQW4=']

In [24]:
bcom_2026_data = ld.get_history(universe=rics, fields=["BID"], start=START_2026, end= END_2026)

bcom_2026_data.head(10)

c:\Project\Code\TRTH_Chain_History\.venv\Lib\site-packages\lseg\data\_tools\_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


BID,91282CDJ7=,91282CDY4=,91282CFF3=,91282CFV8=,91282CGM7=,91282CHC8=,91282CHT1=,91282CJJ1=,91282CJZ5=,91282CKQ3=,91282CLF6=,91282CPQ8=,91282CPY1=,91282CQC8=,91282CQF1=,91282CQN4=,91282CQQ7=,91282CQT1=,91282CQW4=
Date,,,,,,,,,,,,,,,,,,,
2026-05-01,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,97.875,101.71875,98.351562,<NA>,97.15625,98.273438,98.960938,97.460938,100.375,99.59375,<NA>,<NA>,<NA>
2026-05-04,86.398438,88.28125,91.992188,99.429688,95.65625,94.664062,97.53125,101.34375,97.960938,100.34375,96.75,97.9375,98.601562,97.117188,99.976562,99.234375,<NA>,<NA>,<NA>
2026-05-05,86.398438,88.28125,91.992188,99.414062,95.65625,94.664062,97.523438,101.34375,97.96875,100.375,96.78125,97.929688,98.601562,97.117188,100.023438,99.226562,<NA>,<NA>,<NA>
2026-05-06,86.773438,88.6875,92.414062,99.867188,96.101562,95.125,98.0,101.84375,98.492188,100.90625,97.3125,98.382812,99.0625,97.570312,100.484375,99.703125,<NA>,<NA>,<NA>
2026-05-07,86.617188,88.5,92.21875,99.65625,95.898438,94.914062,97.78125,101.617188,98.25,100.65625,97.0625,98.179688,98.851562,97.367188,100.273438,99.476562,<NA>,<NA>,<NA>
2026-05-08,86.757812,88.65625,92.375,99.820312,96.070312,95.09375,97.96875,101.8125,98.4375,100.84375,97.257812,98.367188,99.03125,97.539062,100.445312,99.664062,<NA>,<NA>,<NA>
2026-05-11,86.492188,88.375,92.070312,99.5,95.75,94.757812,97.609375,101.4375,98.070312,100.46875,96.882812,98.023438,98.6875,97.195312,100.09375,99.3125,<NA>,<NA>,<NA>
2026-05-12,86.25,88.132812,91.804688,99.210938,95.453125,94.460938,97.289062,101.09375,97.742188,100.125,96.539062,97.71875,98.375,96.898438,99.78125,99.0,<NA>,<NA>,<NA>
2026-05-13,86.3125,88.1875,91.875,99.257812,95.5,94.507812,97.34375,101.15625,97.78125,100.15625,96.570312,97.78125,98.4375,96.945312,99.84375,99.054688,99.265625,<NA>,<NA>


In [25]:
ld.close_session()

In [26]:
try:
    logout_response = session.post(
        f"{BASE_URL}/Authentication/InvalidateToken",
        json={"Token": {"Token": token}},
        timeout=TIMEOUT,
    )
    print(f"Logout Tick History response: {logout_response.status_code}, {logout_response.text}")
    logout_response.raise_for_status()
except requests.exceptions.RequestException as exc:
    raise RuntimeError("DSS InvalidateToken request failed") from exc


Logout Tick History response: 200, {"@odata.context":"https://selectapi.datascope.lseg.com/RestApi/v1/$metadata#Edm.String","value":"Token has been invalidated successfully."}
